In [ ]:
import pandas as pd

df = pd.read_excel("cities20.xlsx", index_col=0)

In [ ]:
# Percentilisek számítása - BŐVÍTETT VERZIÓ

def calculate_percentiles(df):
    percentiles = {}
    
    # Földrajzi percentilek
    geo_cols = [col for col in df.columns if '_per_100k' in col and 'geo_' in col]
    for col in geo_cols:
        df[f'{col}_percentile'] = df[col].rank(pct=True)
        percentiles[col.replace('geo_', '').replace('_per_100k', '')] = df[f'{col}_percentile'].to_dict()
    
    # Életstílus percentilek
    lifestyle_cols = [col for col in df.columns if '_per_100k' in col and 'eletstilus_' in col]
    for col in lifestyle_cols:
        df[f'{col}_percentile'] = df[col].rank(pct=True)
        percentiles[col.replace('eletstilus_', '').replace('_per_100k', '')] = df[f'{col}_percentile'].to_dict()
    
    # Turista sűrűség percentil
    if 'crowding_per_100k' in df.columns:
        df['crowding_percentile'] = df['crowding_per_100k'].rank(pct=True)
        percentiles['crowding'] = df['crowding_percentile'].to_dict()
    
    # Ár percentilis - Numbeo adatok alapján
    price_cols = [col for col in df.columns if col.startswith('col_') and pd.api.types.is_numeric_dtype(df[col])]
    if price_cols:
        # Átlagos ár index számítása (egyszerűsítve)
        df['avg_price_index'] = df[price_cols].mean(axis=1, skipna=True)
        df['price_percentile'] = 1- df['avg_price_index'].rank(pct=True)
        percentiles['price'] = df['price_percentile'].to_dict()
    
    # Klíma percentilis - átlaghőmérséklet alapján
    climate_cols = [col for col in df.columns if col.startswith('climate_temp_mean_')]
    if climate_cols:
        # Éves átlaghőmérséklet számítása
        df['annual_avg_temp'] = df[climate_cols].mean(axis=1, skipna=True)
        df['climate_percentile'] = df['annual_avg_temp'].rank(pct=True)
        percentiles['climate'] = df['climate_percentile'].to_dict()
    
    # Távolság percentilis
    if 'distance' in df.columns:
        df['distance_percentile'] = df['distance'].rank(pct=True)
        # Fordítva, mert a kisebb távolság jobb
        df['distance_percentile'] = 1 - df['distance_percentile']
        percentiles['distance'] = df['distance_percentile'].to_dict()
    
    return percentiles

# Percentilek számítása
percentile_data = calculate_percentiles(df)

# Cities dictionary frissítése percentilis értékekkel
cities = {}
for city in df.index:
    cities[city] = {
        "földrajz": {
            "tengerpart": percentile_data.get('beach', {}).get(city, 0.5),
            "hegy": percentile_data.get('mountain', {}).get(city, 0.5),
            "város": 0.7,  # placeholder vagy más metrika
            "sziget": percentile_data.get('island', {}).get(city, 0.5),
            "tópart": percentile_data.get('lake', {}).get(city, 0.5),
            "sivatag": percentile_data.get('desert', {}).get(city, 0.1)
        },
        "ár": percentile_data.get('price', {}).get(city, 0.5),
        "klíma": percentile_data.get('climate', {}).get(city, 0.5),
        "életstílus": {
            "bulis": percentile_data.get('bulis', {}).get(city, 0.5),
            "relax": percentile_data.get('relax', {}).get(city, 0.5),
            "aktív": 0.5,  # placeholder
            "kulturális": percentile_data.get('kulturalis', {}).get(city, 0.5),
            "családbarát": percentile_data.get('csaladbarat', {}).get(city, 0.5)
        },
        "távolság": percentile_data.get('distance', {}).get(city, 0.5),
        "zsúfoltság": percentile_data.get('crowding', {}).get(city, 0.5)
    }

# Ellenőrzés - első néhány város adatainak megjelenítése
for city in list(df.index)[:3]:
    print(f"\n{city} percentilis értékek:")
    print(f"  Ár: {cities[city]['ár']:.3f}")
    print(f"  Klíma: {cities[city]['klíma']:.3f}")
    print(f"  Távolság: {cities[city]['távolság']:.3f}")

In [ ]:
# User input (csúszkák, multi-select)
user_preferences = {
    "súlyok": {  # nulladik kérdés: mennyire fontos az egyes dimenzió
        "földrajz": 8,
        "ár": 9,
        "klíma": 7,
        "életstílus": 10,
        "távolság": 6,
        "zsúfoltság": 5,
    },
    "földrajz": {"tengerpart": 8, "hegy": 3, "város":5, "sziget":7, "tópart":2, "sivatag":1},
    "ár": 0.9,         
    "klíma": 0.8,      
    "életstílus": {"bulis":4, "relax":8, "aktív":6, "kulturális":7, "családbarát":5},
    "távolság": 0.9,   
    "zsúfoltság": 0.7,
}


# Normalizált súlyok (összeg=1)
total_weight = sum(user_preferences["súlyok"].values())
weights = {k: v/total_weight for k, v in user_preferences["súlyok"].items()}

# Simple similarity function
def similarity(user_val, city_val):
    return 1 - abs(user_val - city_val)

# Weighted similarity per city
def city_score(user_pref, city_data, weights):
    total = 0
    for attr, weight in weights.items():
        if attr in ["földrajz", "életstílus"]:
            # átlag a multi-select értékekből
            user_vals = user_pref[attr]
            city_vals = city_data[attr]
            sim_vals = []
            for k in user_vals.keys():
                sim_vals.append(similarity(user_vals[k]/10, city_vals[k]))
            avg_sim = sum(sim_vals)/len(sim_vals)
            total += weight * avg_sim
        else:
            # EGYÉNI érték
            user_val = user_pref[attr]
            city_val = city_data[attr]
            sim = similarity(user_val, city_val)
            total += weight * sim
    return total


# Calculate scores for all cities
city_scores = {}
for city_name, city_data in cities.items():
    score = city_score(user_preferences, city_data, weights)
    city_scores[city_name] = score

# Sort top cities
top_cities = sorted(city_scores.items(), key=lambda x: x[1], reverse=True)

# Output
print("Top ajánlott városok a preferenciáid alapján:")
for city, score in top_cities:
    print(f"{city}: {score:.3f}")